<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://learningjournal.github.io/pub-resources/logos/scholarnest_academy.jpg" alt="ScholarNest Academy" style="width: 1400px">
</div>

####1. Requirement
Read data from students_offline.csv file and load into offline_students_raw table.

In [0]:
offline_students_schema = "id string, first_name string, last_name string, address string, skills string, contacts string"

offline_students_raw_df = (
    spark.read.format("csv")
        .option("header", "true")
        .option("quote", "\"")
        .option("escape", "\"")
        .schema(offline_students_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/students_offline.csv")
)

#offline_students_raw_df.display()
offline_students_raw_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.spark_db.offline_students_raw")

####2. Requirement
Prepare an offline_var_students table which is ready for analysis


In [0]:
from pyspark.sql.functions import parse_json

offline_students_df = (
    offline_students_raw_df.withColumns({
        "address": parse_json("address"),
        "skills": parse_json("skills"),
        "contacts": parse_json("contacts")
    })
)

#offline_students_df.display()
offline_students_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_var_students")

####3. Requirement
Perform the following analysis
1. What is country wise student count.
2. Find all students with more than 1 years of Spark knowledge
3. Find all students who didn't provide phone or whatsapp

2.1 What is country wise student count.

In [0]:
%sql
-- Variant object element names are case sensitive

select cast(address:Country as string), count(*) as count
from dev.spark_db.offline_var_students
group by cast(address:Country as string)

Country,count
India,3
Engaland,1
Northern Ireland,1


2.2 Find all students with more than 1 years of Spark knowledge

In [0]:
%sql

with offline_students_skills(
  select id, first_name, last_name, cast(value:Skill as string), cast(value:YearsOfExperience as int)
  from dev.spark_db.offline_var_students, lateral variant_explode_outer(skills)
)
select *
from offline_students_skills
where skill like "%Spark%" and yearsofexperience>1

id,first_name,last_name,Skill,YearsOfExperience
101,Prashant,Pandey,Apache Spark,5
104,Nasima,Khatun,Apache Spark,2
105,Pritam,Jain,Apache Spark,3


2.3 Find all students who didn't provide phone or whatsapp

In [0]:
%sql

select id, first_name, last_name, contacts:email
from dev.spark_db.offline_var_students
where contacts:phone is null and contacts:whatsapp is null

id,first_name,last_name,email
103,Katie,Mcloskey,"""ert89@abc.com"""
104,Nasima,Khatun,"""magt23@abc.com"""


&copy; 2021-2026 <a href="https://www.scholarnest.com/">ScholarNest</a>. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation.</a><br/>
Databricks, Databricks Cloud and the Databricks logo are trademarks of the <a href="https://www.databricks.com/">Databricks Inc.</a><br/>
<a href="https://www.scholarnest.com/pages/privacy">Privacy Policy</a> | <a href="https://www.scholarnest.com/pages/terms">Terms of Use</a> | <a href="https://www.scholarnest.com/pages/contact">Contact Us</a>